##### WARNING
The following notebook is intended to be read only. Please do not modify the contents of this notebook.


# Overview
In this notebook, we will create a simple gold model to serve as the basis for the Patient Outreach Analytics Power BI report.  We transform resources in the `silver` lakehouse into Patient Outreach Analytics Gold shape. We will insert the resulting data into POA Gold lakehouse (defined in current notebook as `poa_gold_database_name`).

By default, you are not expected to make any changes to this file. In case you would like to point to different source and target lakehouses, you can make changes in the `msft_config` notebook and to `poa_gold_database_name` variable in this notebook.

### Notebook Job Scheduling:
We recommend scheduling this notebook job to run every 4 hours. Initial run may not have data to consume due to concurrent and dependent jobs, leading to latency. Adjusting the frequency of higher layer jobs can reduce this latency.

### Usage
Execute the notebook.
_For more information and detailed steps see the [Healthcare data solutions Documentation](https://aka.ms/hds-doc)_


##### Configuration management and setup
The following cells will setup and manage configurations for the Healthcare data solutions:

In [ ]:
%run msft_config_notebook

In [ ]:
%run msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

##### Configuration constants

In [ ]:
#Lakehouse/Database config
poa_gold_database_name = "%%poa_gold_database_name%%"

##### Invoke the Healthcare data solutions API

Now we will invoke the APIs to transform data into the `Patient Outreach analytics GOLD` model

`inline_params` is a json dictionary of parameters(configuration values) which will take precedence and be use in place of the configuration values in the administration lakehouse

In [ ]:
inline_params = "{}"

In [ ]:
from microsoft.fabric.hls.hds.services.patient_outreach_gold_ingestion_service import PatientOutreachGoldIngestionService
import json

# convert inline params into dictionary
inline_params_dict = json.loads(inline_params)

patient_outreach_gold_ingestion_service = PatientOutreachGoldIngestionService(spark,
        workspace_name=workspace_name,
        solution_name=solution_name,
        admin_lakehouse_name=administration_database_name,
        inline_params=inline_params_dict,
        one_lake_endpoint=one_lake_endpoint
        )
patient_outreach_gold_ingestion_service.run()

##### Appointments to Journey Event attribution

In the implemented logic below we will attribute appointments to the journey events. 
We will use the following conditions:
- If the appointment scheduled date is within `days_from_journey` days of the event.
- If appointment service type is the same as the event Journey service type.
- The journey event is not already attributed to an appointment.

If your attribution logic is different than the suggested logic, this is the only place where the attribution logic needs to change.

In [ ]:
from pyspark.sql.utils import AnalysisException
from microsoft.fabric.hls.hds.utils.parameter_service import ParameterService
# number of days from event_date to match appointment
# default value is 1 month (30 days)

parameter_service = ParameterService(spark,
    workspace_name = workspace_name,
    admin_lakehouse_name = administration_database_name,
    one_lake_endpoint=one_lake_endpoint)

days_from_journey = int(parameter_service.get_activity_config_value('days_from_journey', 30))

#find all possible attributions
matched_app = spark.sql(f"""
WITH closest_appointments AS (
    SELECT 
        m.MarketingEventKey, 
        a.AppointmentKey,
        ROW_NUMBER() OVER (PARTITION BY m.MarketingEventKey ORDER BY ABS(datediff(a.AppointmentScheduledDate, m.EventDate))) as rn
    FROM `{poa_gold_database_name}`.marketingeventfact m
    JOIN `{poa_gold_database_name}`.journeydim j 
    ON m.JourneyKey = j.JourneyKey
    JOIN `{poa_gold_database_name}`.appointmentdim a 
    ON a.PatientKey = m.PatientKey
        AND a.ServiceLineKey = j.ServiceLineKey
        AND datediff(a.AppointmentScheduledDate, m.EventDate) <= {days_from_journey} 
        AND a.AppointmentScheduledDate >= m.EventDate
    WHERE m.AppointmentKey IS NULL)
SELECT 
    MarketingEventKey, 
    AppointmentKey
FROM closest_appointments
WHERE rn = 1;
""")

print(f"Found {matched_app.count()} eventfacts that could be attributed to appointments.")

if matched_app.count()>0:
    try:
        matched_app.createOrReplaceTempView("temporary_app")
        spark.sql(f"""
            MERGE INTO `{poa_gold_database_name}`.marketingeventfact AS m
            USING temporary_app AS t
            ON m.MarketingEventKey = t.MarketingEventKey
            WHEN MATCHED THEN
            UPDATE SET m.AppointmentKey = t.AppointmentKey
        """)
        print(f"Attribution executed succefully.")
    except AnalysisException as error:
        print(f"Attributionfailed: {error}")
    finally:
        spark.sql("DROP VIEW IF EXISTS temporary_app")

In [ ]:
mssparkutils.fs.unmount(packages_mount_name)